# 1. Discovery and download

This notebook finds spectral grids and filter sets, downloads small ones, and shows where SED_Tools keeps them.
Every later notebook uses what is downloaded here:
- the `bbody` grid (blackbodies; Teff only, 41 MB);
- the `Husfeld` grid (hot stars, split by helium abundance `yhe`; 42 MB);
- a cut of `Kurucz2003all`: Teff 4000–8000 K, log g 3–5, [M/H] −1 to +0.5 (850 spectra, about 63 MB; the whole grid is about 1 GB);
- the filter sets `Generic/Johnson` and `2MASS/2MASS`, and NJM's Vega spectrum.

Downloads come from the NJM mirror first. NJM is optional: without it, filters come from the instrument teams' own files or SVO, and spectral grids from SVO, MSG or MAST (section 5).

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from sed_tools.api import SED, Filters

plt.rcParams["figure.dpi"] = 80
KURUCZ_CUT = dict(teff_min=4000, teff_max=8000, logg_min=3.0, logg_max=5.0,
                  metallicity_min=-1.0, metallicity_max=0.5)

## 1. Where the data goes

Everything lives under one data directory, `~/.sed_tools/data` by default. `sed-tools config --set DIR` or the `SED_DATA_DIR` environment variable moves it.

In [ ]:
!sed-tools config

## 2. Finding spectral grids

`SED.query()` lists the grids every source offers: NJM, SVO, MSG and MAST. Each row is a `SpectralGridInfo` with the grid's parameter ranges.
`source=` asks one source only. The range arguments keep grids that reach into the given ranges.

In [ ]:
njm = SED.query(source="njm")
for info in njm:
    print(f"{info.name:18s} Teff {info.teff_range}  log g {info.logg_range}  [M/H] {info.metallicity_range}")

In [ ]:
solar = SED.query(source="njm", teff_min=5000, teff_max=7000, logg_min=4.0)
print([info.name for info in solar])

kurucz = next(info for info in njm if info.name == "Kurucz2003all")
print(kurucz.covers(teff=5777, logg=4.44), kurucz.covers_range(teff_min=4000, teff_max=8000))

## 3. Downloading spectral grids

`SED.fetch(name)` downloads a grid and returns its spectra (`.grid`); `SED.local(name)` then loads it for interpolation (notebook 4). With `teff_min` … `metallicity_max`, only the spectra inside the bounds are downloaded (NJM, MSG and MAST; SVO downloads the whole grid, then cuts it).
A fetch that finds every spectrum already present downloads nothing, so running these cells again is cheap.

Each download ends by printing the line(s) to give MESA's `&colors` inlist as `stellar_atm`. A grid split into variants (here `Husfeld`, by `yhe`, and `Kurucz2003all`, by `alpha`) prints one line per variant: MESA reads one variant folder, never the grid's own folder (notebook 6).

In [ ]:
bbody = SED.fetch("bbody")
husfeld = SED.fetch("Husfeld")

In [ ]:
kurucz = SED.fetch("Kurucz2003all", **KURUCZ_CUT)
print(len(kurucz.grid), "spectra fetched")

## 4. Filters and the Vega spectrum

`Filters.fetch(facility, instrument)` installs a filter set: one `.dat` curve per band, the MESA index file named after the instrument, and `manifest.json`, `properties.json` and `info.json`. It returns the folder and prints its `instrument = '...'` inlist line.
`Filters.fetch_vega()` installs NJM's Vega spectrum, which Vega magnitudes need (MESA's `vega_sed`).

In [ ]:
johnson = Filters.fetch("Generic", "Johnson")
twomass = Filters.fetch("2MASS", "2MASS")
vega_path = Filters.fetch_vega()
print(sorted(p.name for p in johnson.iterdir()))
print(vega_path)

In [ ]:
print(Filters.query(include_remote=False))
remote = Filters.query(facility="2MASS", include_local=False)
print([(row["facility_key"], row["source"]) for row in remote])

## 5. NJM is optional

The order of sources is:
- **Filters:** the installed copy, then NJM, then the instrument team's original files (the original-source builders), then the SVO Filter Profile Service.
- **Spectral grids:** NJM, then SVO, MSG and MAST. `SED.fetch(name, source="svo")` forces one source.

When NJM does not answer within 5 s, SED_Tools warns once, uses the other sources, and leaves NJM alone for five minutes.
The package ships its own instrument catalogue, so names and descriptions are the same with or without NJM.

NJM's catalogue files (`index.json`, `properties.json`, `info.json`) are cached beside the filter root. `offline=True` reads only installed and cached copies and never contacts NJM; `include_remote=False` does the same for availability checks.

In [ ]:
info = Filters.info("2MASS", "2MASS", offline=True)
print(info["licence"])
available = Filters.available("2MASS", "2MASS", include_remote=False)
print(available["published"], available["components"], available["detector_type"])

In [ ]:
svo = SED.query(source="svo")
print(len(svo), "grids on SVO, for example", [info.name for info in svo[:8]])

## 6. Installed grids

`SED.query(include_remote=False)` lists what is installed. A grid split into variants is listed by its variant folders (`Kurucz2003all/Kurucz2003all_alpha_…`); `SED.local("Kurucz2003all")` still loads the grid as a whole, with `alpha` as a fourth axis (notebook 4).

In [ ]:
for info in SED.query(include_remote=False):
    print(f"{info.name:55s} {info.n_spectra:5d} spectra")

## Command-line equivalents

| Python | Command line |
|---|---|
| `SED.query()` | `sed-tools grids` (interactive list) |
| `SED.fetch("Kurucz2003all", teff_min=4000, …)` | `sed-tools grids --grids Kurucz2003all --teff 4000 8000 --logg 3 5 --meta -1 0.5` |
| `SED.fetch(name, source="svo")` | `sed-tools grids --grids NAME --source svo` |
| `Filters.fetch("2MASS", "2MASS")` | `sed-tools filters 2MASS/2MASS` |
| `Filters.fetch_vega()` | `sed-tools vega` |
| `Filters.info(..., offline=True)` | `sed-tools filters-info 2MASS/2MASS --offline` |
| data directory | `sed-tools config`, `sed-tools config --set DIR --move` |

`sed-tools` with no arguments opens the interactive menu.

In [ ]:
!sed-tools filters-info 2MASS/2MASS --offline